# VinDr-SpineXR — Dataset EDA (Slice 2)
Exploratory analysis of DICOM images and bounding-box annotations.

In [ ]:
# ─── Section 1: Imports and path config ───────────────────────────────────────
import os, sys, warnings, tempfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend; change to 'TkAgg' if running locally
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import pydicom
warnings.filterwarnings('ignore')

# ── Project root on path ──────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Data paths — edit these for real data ─────────────────────────────────────
DATA_DIR      = '/path/to/vindr-spinexr'        # root of downloaded dataset
CSV_FILE      = os.path.join(DATA_DIR, 'annotations', 'train.csv')
IMG_DIR       = os.path.join(DATA_DIR, 'train_images')
REPORTS_DIR   = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

# ── EDA config ────────────────────────────────────────────────────────────────
N_SAMPLE_IMAGES = 4    # DICOMs to display in section 5
ROI_SIZES       = [28, 64, 96]
SEED            = 42
np.random.seed(SEED)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'CSV file     : {CSV_FILE}')
print(f'Image dir    : {IMG_DIR}')

In [ ]:
# ─── Section 2: Data availability — real vs. synthetic stub ───────────────────
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian
import pydicom.uid

USING_SYNTHETIC = False
_tmp_dir = None

def _generate_synthetic_stub():
    """Mirrors the stub generator from experiments/smoke_test_loader.py."""
    tmp = tempfile.mkdtemp(prefix='qstrata_eda_')
    csv_data = {
        'image_id': ['img_001','img_001','img_002','img_002','img_003',
                     'img_003','img_004','img_004','img_005','img_005'],
        'xmin':     [10, 50, 20, 80, 15,  60, 30,  90, 25,  70],
        'ymin':     [10, 50, 20, 80, 15,  60, 30,  90, 25,  70],
        'xmax':     [60,100, 70,130, 65, 110, 80, 140, 75, 120],
        'ymax':     [60,100, 70,130, 65, 110, 80, 140, 75, 120],
        'class_id': [1, 2, 1, 3, 2, 1, 3, 2, 1, 3],
    }
    csv_path = os.path.join(tmp, 'stub_annotations.csv')
    pd.DataFrame(csv_data).to_csv(csv_path, index=False)

    rng = np.random.default_rng(seed=42)
    for img_id in ['img_001','img_002','img_003','img_004','img_005']:
        pa = rng.integers(100, 2001, size=(512,512), dtype=np.uint16)
        fm = FileMetaDataset()
        fm.MediaStorageSOPClassUID    = pydicom.uid.DigitalXRayImageStorageForPresentation
        fm.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
        fm.TransferSyntaxUID          = ExplicitVRLittleEndian
        ds = FileDataset(None, {}, file_meta=fm, preamble=b'\x00'*128)
        ds.is_implicit_VR = False; ds.is_little_endian = True
        ds.Rows=512; ds.Columns=512; ds.BitsAllocated=16; ds.BitsStored=16
        ds.HighBit=15; ds.PixelRepresentation=0; ds.SamplesPerPixel=1
        ds.PhotometricInterpretation='MONOCHROME2'
        ds.RescaleSlope=1.0; ds.RescaleIntercept=-1000.0
        ds.PixelData = pa.tobytes()
        ds.save_as(os.path.join(tmp, f'{img_id}.dicom'))
    return csv_path, tmp

if not (os.path.exists(CSV_FILE) and os.path.isdir(IMG_DIR)):
    print('[INFO] Real data not found — running on SYNTHETIC STUB.')
    print('[INFO] Download real data: https://physionet.org/content/vindr-spinexr/1.0.0/')
    CSV_FILE, IMG_DIR = _generate_synthetic_stub()
    _tmp_dir = os.path.dirname(CSV_FILE)
    USING_SYNTHETIC = True
    print(f'[INFO] Stub directory: {_tmp_dir}')
else:
    print('[INFO] Real VinDr-SpineXR data detected. Running full EDA.')

print(f'Data source  : {"SYNTHETIC STUB" if USING_SYNTHETIC else "REAL DATA"}')
print(f'CSV          : {CSV_FILE}')
print(f'Image dir    : {IMG_DIR}')

In [ ]:
# ─── Section 3: CSV inspection ────────────────────────────────────────────────
REQUIRED_COLS = ['image_id', 'xmin', 'ymin', 'xmax', 'ymax', 'class_id']

df = pd.read_csv(CSV_FILE)

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f'[FAIL] Missing required columns: {missing_cols}')

print('=== CSV Inspection ===')
print(f'  Shape         : {df.shape}')
print(f'  Columns       : {list(df.columns)}')
print()
print('  Dtypes:')
print(df.dtypes.to_string(header=False))
print()
print('  Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else '    None')
print()
n_dup = df.duplicated().sum()
print(f'  Duplicate rows : {n_dup}')
print(f'  Unique image_id: {df["image_id"].nunique()}')
print()
print('  Head (5 rows):')
display(df.head())

In [ ]:
# ─── Section 4: Class distribution ───────────────────────────────────────────
# Drop rows without valid bbox for annotation-level analysis
ann_df = df.dropna(subset=['xmin','ymin','xmax','ymax']).reset_index(drop=True)

class_counts = ann_df['class_id'].value_counts().sort_index()
class_pct    = (class_counts / class_counts.sum() * 100).round(2)
imbalance_ratio = class_counts.max() / class_counts.min()

print('=== Class Distribution ===')
summary = pd.DataFrame({'count': class_counts, 'pct': class_pct})
print(summary.to_string())
print(f'\n  Total annotations : {len(ann_df)}')
print(f'  Imbalance ratio   : {imbalance_ratio:.2f}x  (max/min class)')
IMBALANCE_WARNING = imbalance_ratio > 3.0
if IMBALANCE_WARNING:
    print(f'  [WARN] Imbalance ratio {imbalance_ratio:.1f}x > 3.0 — consider weighted sampling or augmentation.')
else:
    print('  [OK] Imbalance ratio within acceptable range (<= 3.0).')

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(class_counts.index.astype(str), class_counts.values,
              color='steelblue', edgecolor='k', linewidth=0.6)
for bar, pct in zip(bars, class_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_title('Annotation Class Distribution')
ax.set_xlabel('class_id')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'class_distribution.png'), dpi=120)
plt.show()
print('  Plot saved → reports/class_distribution.png')

In [ ]:
# ─── Section 5: Bounding box statistics ──────────────────────────────────────
ann_df = ann_df.copy()
ann_df['bb_w']    = ann_df['xmax'] - ann_df['xmin']
ann_df['bb_h']    = ann_df['ymax'] - ann_df['ymin']
ann_df['bb_area'] = ann_df['bb_w'] * ann_df['bb_h']
ann_df['aspect']  = (ann_df['bb_w'] / ann_df['bb_h'].replace(0, np.nan)).round(3)

bb_stats = ann_df[['bb_w','bb_h','bb_area','aspect']].describe().round(2)

print('=== Bounding Box Statistics ===')
print(bb_stats.to_string())

# Recommended ROI size: next power-of-2-ish above median diagonal
median_w = ann_df['bb_w'].median()
median_h = ann_df['bb_h'].median()
median_diag = np.sqrt(median_w**2 + median_h**2)
# Round up to nearest value in [28, 64, 96, 128]
_roi_candidates = [28, 64, 96, 128]
RECOMMENDED_ROI = next((r for r in _roi_candidates if r >= max(median_w, median_h)), 128)
print(f'\n  Median box size : {median_w:.1f} x {median_h:.1f} px')
print(f'  Recommended ROI : {RECOMMENDED_ROI}px  (smallest standard size >= max(median_w, median_h))')

invalid_boxes = ann_df[(ann_df['bb_w'] <= 0) | (ann_df['bb_h'] <= 0)]
print(f'  Invalid boxes (w<=0 or h<=0): {len(invalid_boxes)}')

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, col, title in zip(axes,
                          ['bb_w','bb_h','bb_area','aspect'],
                          ['Box Width','Box Height','Box Area','Aspect Ratio']):
    data = ann_df[col].dropna()
    ax.hist(data, bins=min(30, len(data)), color='coral', edgecolor='k', linewidth=0.5)
    ax.axvline(data.median(), color='navy', lw=1.5, linestyle='--', label=f'median={data.median():.1f}')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Bounding Box Statistics', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'bbox_statistics.png'), dpi=120)
plt.show()
print('  Plot saved → reports/bbox_statistics.png')

In [ ]:
# ─── Section 6: Random DICOM loading — images + metadata ─────────────────────
unique_ids = ann_df['image_id'].unique()
sample_ids = np.random.choice(unique_ids, size=min(N_SAMPLE_IMAGES, len(unique_ids)), replace=False)

def _load_dicom(img_dir, image_id):
    for ext in ('.dicom', '.dcm'):
        p = os.path.join(img_dir, f'{image_id}{ext}')
        if os.path.exists(p):
            return pydicom.dcmread(p), p
    return None, None

def _normalize(arr):
    lo, hi = arr.min(), arr.max()
    if hi > lo:
        return (arr - lo) / (hi - lo)
    return arr

print('=== Random DICOM Samples ===')
fig, axes = plt.subplots(1, len(sample_ids), figsize=(4*len(sample_ids), 4))
if len(sample_ids) == 1:
    axes = [axes]

loaded_ok = []
for ax, sid in zip(axes, sample_ids):
    ds, path = _load_dicom(IMG_DIR, sid)
    if ds is None:
        ax.set_title(f'{sid}\n[NOT FOUND]', fontsize=8)
        ax.axis('off')
        continue
    pa = ds.pixel_array.astype(np.float32)
    if pa.ndim == 3:
        pa = pa.mean(axis=0)
    if hasattr(ds,'RescaleSlope') and hasattr(ds,'RescaleIntercept'):
        pa = pa * float(ds.RescaleSlope) + float(ds.RescaleIntercept)
    norm = _normalize(pa)
    ax.imshow(norm, cmap='gray')
    ax.set_title(f'{sid}\n{pa.shape[1]}x{pa.shape[0]}', fontsize=8)
    ax.axis('off')
    loaded_ok.append(sid)
    print(f'  {sid} — shape:{pa.shape} | ')
    for attr in ['Rows','Columns','BitsAllocated','PhotometricInterpretation','RescaleSlope','RescaleIntercept']:
        val = getattr(ds, attr, 'N/A')
        print(f'      {attr}: {val}')

plt.suptitle('Random DICOM Samples (normalised)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'dicom_samples.png'), dpi=120)
plt.show()
print(f'  Loaded {len(loaded_ok)}/{len(sample_ids)} images OK.')
print('  Plot saved → reports/dicom_samples.png')

In [ ]:
# ─── Section 7: Bbox overlay visualisation ────────────────────────────────────
CLASS_COLOURS = {0:'white', 1:'lime', 2:'cyan', 3:'yellow', 4:'orange', 5:'red'}

overlay_ids = np.random.choice(unique_ids, size=min(4, len(unique_ids)), replace=False)

fig, axes = plt.subplots(1, len(overlay_ids), figsize=(4*len(overlay_ids), 4))
if len(overlay_ids) == 1:
    axes = [axes]

print('=== Bounding Box Overlays ===')
for ax, sid in zip(axes, overlay_ids):
    ds, _ = _load_dicom(IMG_DIR, sid)
    if ds is None:
        ax.set_title(f'{sid}\n[NOT FOUND]', fontsize=8); ax.axis('off')
        continue
    pa = ds.pixel_array.astype(np.float32)
    if pa.ndim == 3: pa = pa.mean(axis=0)
    if hasattr(ds,'RescaleSlope'): pa = pa * float(ds.RescaleSlope) + float(ds.RescaleIntercept)
    ax.imshow(_normalize(pa), cmap='gray')

    rows_for_id = ann_df[ann_df['image_id'] == sid]
    for _, row in rows_for_id.iterrows():
        cls  = int(row['class_id'])
        col  = CLASS_COLOURS.get(cls, 'magenta')
        rect = mpatches.Rectangle(
            (row['xmin'], row['ymin']),
            row['xmax'] - row['xmin'],
            row['ymax'] - row['ymin'],
            linewidth=1.5, edgecolor=col, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(row['xmin'], row['ymin'] - 4, f'cls={cls}',
                color=col, fontsize=7, fontweight='bold')
    ax.set_title(sid, fontsize=8)
    ax.axis('off')
    print(f'  {sid}: {len(rows_for_id)} annotations')

plt.suptitle('Annotation Overlays', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'bbox_overlays.png'), dpi=120)
plt.show()
print('  Plot saved → reports/bbox_overlays.png')

In [ ]:
# ─── Section 8: ROI extraction preview ───────────────────────────────────────
# Pick one annotation to preview at multiple scales
sample_row = ann_df.iloc[0]
sid = sample_row['image_id']
ds, _ = _load_dicom(IMG_DIR, sid)

print('=== ROI Extraction Preview ===')

if ds is None:
    print(f'[WARN] Cannot preview ROI — DICOM not found for {sid}')
else:
    pa = ds.pixel_array.astype(np.float32)
    if pa.ndim == 3: pa = pa.mean(axis=0)
    if hasattr(ds,'RescaleSlope'): pa = pa * float(ds.RescaleSlope) + float(ds.RescaleIntercept)
    norm_full = _normalize(pa)

    xmin = max(0, int(sample_row['xmin']))
    ymin = max(0, int(sample_row['ymin']))
    xmax = min(pa.shape[1], int(sample_row['xmax']))
    ymax = min(pa.shape[0], int(sample_row['ymax']))
    crop_raw = norm_full[ymin:ymax, xmin:xmax]

    if crop_raw.size == 0:
        print(f'[WARN] Zero-area crop for {sid} — box coords: {xmin},{ymin},{xmax},{ymax}')
    else:
        # CLAHE-enhanced crop
        crop_8u  = (crop_raw * 255).astype(np.uint8)
        clahe    = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4,4))
        crop_cl  = clahe.apply(crop_8u)

        # Multi-scale resized crops
        col_count = 2 + len(ROI_SIZES)   # raw, clahe, + scaled versions
        fig, axes = plt.subplots(1, col_count, figsize=(3*col_count, 3))

        axes[0].imshow(crop_raw, cmap='gray'); axes[0].set_title('Raw crop'); axes[0].axis('off')
        axes[1].imshow(crop_cl,  cmap='gray'); axes[1].set_title('CLAHE'); axes[1].axis('off')

        for ax, sz in zip(axes[2:], ROI_SIZES):
            resized = cv2.resize(crop_cl, (sz, sz)).astype(np.float32) / 255.0
            ax.imshow(resized, cmap='gray')
            ax.set_title(f'{sz}px')
            ax.axis('off')
            print(f'  ROI {sz}px — min:{resized.min():.3f}  max:{resized.max():.3f}  mean:{resized.mean():.3f}')

        plt.suptitle(f'ROI Extraction: {sid}  cls={int(sample_row["class_id"])}', fontsize=10)
        plt.tight_layout()
        plt.savefig(os.path.join(REPORTS_DIR, 'roi_preview.png'), dpi=120)
        plt.show()
        print('  Plot saved → reports/roi_preview.png')

In [ ]:
# ─── Section 9: Quality checks ────────────────────────────────────────────────
print('=== Quality Checks ===')

# 1. Missing DICOMs
def _dicom_exists(img_dir, image_id):
    for ext in ('.dicom', '.dcm'):
        if os.path.exists(os.path.join(img_dir, f'{image_id}{ext}')):
            return True
    return False

all_ids   = ann_df['image_id'].unique()
missing_dicoms = [i for i in all_ids if not _dicom_exists(IMG_DIR, i)]
MISSING_DICOM_WARNING = len(missing_dicoms) > 0
print(f'  Missing DICOMs    : {len(missing_dicoms)}/{len(all_ids)} unique image IDs')
if missing_dicoms[:5]:
    print(f'  Examples          : {missing_dicoms[:5]}')

# 2. Invalid bounding boxes
invalid_bb = ann_df[(ann_df['bb_w'] <= 0) | (ann_df['bb_h'] <= 0)]
print(f'  Invalid boxes     : {len(invalid_bb)}  (width or height <= 0)')

# 3. Zero-area crops
zero_area = ann_df[ann_df['bb_area'] <= 0]
print(f'  Zero-area crops   : {len(zero_area)}')

# 4. Zero-patch rate (load + extract for all items)
import torch
from qcore.data.spine_dataset import SpineCascadeDataset

ps = RECOMMENDED_ROI
dataset = SpineCascadeDataset(CSV_FILE, IMG_DIR, patch_size=(ps, ps))
n_zeros = 0
for i in range(len(dataset)):
    patch, _ = dataset[i]
    if torch.norm(patch) == 0.0:
        n_zeros += 1

zero_patch_pct = 100.0 * n_zeros / len(dataset)
print(f'  Zero-patch rate   : {n_zeros}/{len(dataset)} ({zero_patch_pct:.1f}%)')
if zero_patch_pct > 10.0:
    print(f'  [WARN] High zero-patch rate — check DICOM paths or box coords.')

# 5. NaN in key columns
nan_rows = df[df[['xmin','ymin','xmax','ymax','class_id']].isnull().any(axis=1)]
print(f'  Rows with NaN in key cols: {len(nan_rows)}')

print()
QUALITY_ISSUES = []
if MISSING_DICOM_WARNING:   QUALITY_ISSUES.append(f'{len(missing_dicoms)} missing DICOMs')
if len(invalid_bb) > 0:     QUALITY_ISSUES.append(f'{len(invalid_bb)} invalid boxes')
if len(zero_area) > 0:      QUALITY_ISSUES.append(f'{len(zero_area)} zero-area crops')
if zero_patch_pct > 10.0:   QUALITY_ISSUES.append(f'High zero-patch rate ({zero_patch_pct:.1f}%)')
if len(nan_rows) > 0:       QUALITY_ISSUES.append(f'{len(nan_rows)} NaN annotation rows')
print(f'  Quality issues    : {QUALITY_ISSUES if QUALITY_ISSUES else "NONE"}')

In [ ]:
# ─── Section 10: EDA Summary + GO/NO-GO + report generation ──────────────────
print('=== EDA Summary ===')

GO_NOGO = 'GO'
if MISSING_DICOM_WARNING and len(missing_dicoms) / len(all_ids) > 0.10:
    GO_NOGO = 'NO-GO'
if len(invalid_bb) > 0:
    GO_NOGO = 'NO-GO'
if USING_SYNTHETIC:
    print('  [NOTE] Running on synthetic stub — all quality checks structural only.')
    print('  [NOTE] Re-run with real VinDr-SpineXR data for a meaningful GO/NO-GO.')

print(f'  Recommended ROI size       : {RECOMMENDED_ROI}px')
print(f'  Class imbalance warning    : {"YES" if IMBALANCE_WARNING else "NO"}')
print(f'  Missing DICOM warning      : {"YES" if MISSING_DICOM_WARNING else "NO"}')
print(f'  Quality issues found       : {QUALITY_ISSUES if QUALITY_ISSUES else "NONE"}')
print(f'  GO for classification      : {GO_NOGO}')

# ── Write reports/eda_summary.md ─────────────────────────────────────────────
n_classes = len(ann_df['class_id'].unique())
class_balance_md = '\n'.join(
    f'| {cls} | {cnt} | {pct:.1f}% |'
    for cls, cnt, pct in zip(class_counts.index, class_counts.values, class_pct.values)
)

report = f"""# VinDr-SpineXR EDA Summary

**Data source:** {'SYNTHETIC STUB (real data not present)' if USING_SYNTHETIC else 'REAL VinDr-SpineXR data'}

---

## Dataset Statistics

| Metric | Value |
|--------|-------|
| Total CSV rows | {len(df)} |
| Valid annotations (with bbox) | {len(ann_df)} |
| Unique image IDs | {ann_df['image_id'].nunique()} |
| Number of classes | {n_classes} |
| Duplicate rows | {n_dup} |
| NaN annotation rows | {len(nan_rows)} |

## Class Balance

| class_id | count | pct |
|----------|-------|-----|
{class_balance_md}

**Imbalance ratio:** {imbalance_ratio:.2f}x (max/min class)
**Imbalance warning (>3x):** {'YES' if IMBALANCE_WARNING else 'NO'}

## Bounding Box Statistics

| Metric | Width | Height | Area | Aspect |
|--------|-------|--------|------|--------|
| mean | {ann_df['bb_w'].mean():.1f} | {ann_df['bb_h'].mean():.1f} | {ann_df['bb_area'].mean():.1f} | {ann_df['aspect'].mean():.2f} |
| median | {ann_df['bb_w'].median():.1f} | {ann_df['bb_h'].median():.1f} | {ann_df['bb_area'].median():.1f} | {ann_df['aspect'].median():.2f} |
| std | {ann_df['bb_w'].std():.1f} | {ann_df['bb_h'].std():.1f} | {ann_df['bb_area'].std():.1f} | {ann_df['aspect'].std():.2f} |
| min | {ann_df['bb_w'].min():.1f} | {ann_df['bb_h'].min():.1f} | {ann_df['bb_area'].min():.1f} | {ann_df['aspect'].min():.2f} |
| max | {ann_df['bb_w'].max():.1f} | {ann_df['bb_h'].max():.1f} | {ann_df['bb_area'].max():.1f} | {ann_df['aspect'].max():.2f} |

## ROI Observations

- Median bounding box: **{ann_df['bb_w'].median():.1f} x {ann_df['bb_h'].median():.1f} px**
- **Recommended ROI size: {RECOMMENDED_ROI}px** (smallest standard size >= max(median_w, median_h))
- CLAHE preprocessing applied for bone contrast enhancement
- Patches flattened to {RECOMMENDED_ROI*RECOMMENDED_ROI}-element 1D vectors for QML input

## Quality Issues

| Check | Result |
|-------|--------|
| Missing DICOMs | {len(missing_dicoms)}/{len(all_ids)} ({100*len(missing_dicoms)/max(1,len(all_ids)):.1f}%) |
| Invalid boxes (w/h <= 0) | {len(invalid_bb)} |
| Zero-area crops | {len(zero_area)} |
| Zero-patch rate | {n_zeros}/{len(dataset)} ({zero_patch_pct:.1f}%) |

**Issues found:** {', '.join(QUALITY_ISSUES) if QUALITY_ISSUES else 'NONE'}

## Recommendations

- Use ROI size **{RECOMMENDED_ROI}px** for initial classification baseline
- {'Apply class-weighted sampling or augmentation — imbalance ratio is {:.1f}x'.format(imbalance_ratio) if IMBALANCE_WARNING else 'Class balance acceptable for initial training run'}
- {'Investigate missing DICOMs before training' if MISSING_DICOM_WARNING else 'DICOM coverage appears complete'}

## GO / NO-GO for Classification Baseline

**{GO_NOGO}**

{'> ⚠️ NO-GO reasons: ' + '; '.join(QUALITY_ISSUES) if GO_NOGO == 'NO-GO' else '> ✅ Loader validated. Proceed to Slice 3: ROI Classification Dataset Builder.'}

{'> ⚠️ NOTE: This report was generated from SYNTHETIC data. Re-run with real VinDr-SpineXR data.' if USING_SYNTHETIC else ''}

---
*Generated by: notebooks/01_dataset_eda.ipynb — QStrata Slice 2*
"""

report_path = os.path.join(REPORTS_DIR, 'eda_summary.md')
with open(report_path, 'w') as f:
    f.write(report)
print(f'  Report written → {report_path}')
print()
print('=== NOTEBOOK RUN COMPLETE ===')